# Mixed-design ANOVA — SPSS `.sav` → Google Drive

**Design supported:** one between-subject factor (two or more groups) × **two** within-subject measurements (pre/post); **wide format** (one row per participant).

**Outputs per selected outcome:** formatted Excel workbook, Word report, and PNG figures in your own Google Drive. Multiple pre/post outcome pairs can be selected in the same run. No data are sent to a third-party service by the code; the `.sav` is uploaded to your Colab session, and final reports are written to the Drive you authorize. Do not use a public/shared Drive folder for participant-level outlier flags.

### Instructions
1. Run the **Install packages** cell.
2. Run **Upload SPSS file**, then choose your `.sav` file.
3. Run **Connect Google Drive** and authorize access.
4. Run the **Analysis functions** cell.
5. Run **Choose variables and analyse**, select group, optional participant ID, pre/post outcome pair(s), and click **Run analysis & save to Google Drive**.

**Statistical details:** The two-time-point split-plot ANOVA is computed from separate between-person (subject means) and within-person (subject changes) error terms. Type III tests with sum-to-zero contrasts are used, including for unequal group sizes. Group × time is the omnibus comparison of changes. Results include partial η², assumption checks, change-score Q–Q plots, descriptive confidence intervals, potential outliers, and exploratory within/between follow-ups with Holm correction **within each comparison family**. Each outcome is analysed independently; there is **no across-outcome multiplicity correction**.

**Missing data:** SPSS-declared user-missing values are treated as missing on import. Complete-case analysis is used for each selected outcome pair. If many participants are incomplete, or participants are nested within schools/sites, use an appropriate multilevel or mixed-effects model instead.

**References:** [statsmodels ANOVA and Type III contrasts](https://www.statsmodels.org/stable/examples/notebooks/generated/interactions_anova.html) · [pyreadstat SAV handling](https://github.com/Roche/pyreadstat) · [Google Colab Drive access](https://research.google.com/colaboratory/faq.html)

## 1. Install packages

In [ ]:
%pip -q install 'pyreadstat>=1.2.8' 'statsmodels>=0.14.0' 'python-docx>=1.1.0' 'XlsxWriter>=3.2.0' 'ipywidgets>=8.1.0' 'scipy>=1.11.0'

## 2. Upload the SPSS `.sav` file

In [ ]:
from pathlib import Path
import pyreadstat
import pandas as pd
from google.colab import files, output
from IPython.display import display, Markdown

output.enable_custom_widget_manager()
print('Select your SPSS .sav file from your computer.')
uploaded_files = files.upload()
sav_names = [name for name in uploaded_files if str(name).lower().endswith('.sav')]
if len(sav_names) == 0:
    raise ValueError('No .sav file was uploaded. Rerun this cell and upload an SPSS .sav file.')
if len(sav_names) > 1:
    print(f'Multiple SAV files uploaded. Using: {sav_names[0]}')
sav_filename = sav_names[0]
sav_path = Path(sav_filename).resolve()
raw_data, spss_meta = pyreadstat.read_sav(str(sav_path),
                                            user_missing=False,
                                            apply_value_formats=False)
print(f'Loaded {sav_filename}: {len(raw_data):,} participant rows and '
      f'{raw_data.shape[1]:,} variables.')
print('SPSS-declared missing codes are converted to missing (NaN) on import.')
print('No recoding of raw numeric group or outcome variables has occurred.')

## 3. Connect Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPORT_ROOT = Path('/content/drive/MyDrive/Mixed_ANOVA_Reports')
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('Reports will be written under:', REPORT_ROOT)

## 4. Analysis and reporting functions

In [ ]:
import math
import re
import sys
import json
import platform
from pathlib import Path
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches, Pt, RGBColor


def variable_dictionary(df, meta):
    labels = getattr(meta, 'column_names_to_labels', None) or {}
    if not labels and hasattr(meta, 'column_names') and hasattr(meta, 'column_labels'):
        labels = dict(zip(meta.column_names, meta.column_labels))
    vals = getattr(meta, 'variable_value_labels', {}) or {}
    measure = getattr(meta, 'variable_measure', {}) or {}
    out = []
    for col in df.columns:
        a = df[col]
        value_labels = vals.get(col, {}) or {}
        out.append({
            'Variable': col, 'SPSS label': labels.get(col) or '',
            'Type': str(a.dtype), 'Measure': measure.get(col) or '',
            'Nonmissing (loaded)': int(a.notna().sum()),
            'Missing (loaded)': int(a.isna().sum()),
            'Distinct nonmissing': int(a.nunique(dropna=True)),
            'Value labels': '; '.join(f'{k}: {v}' for k, v in value_labels.items())[:3000]
        })
    return pd.DataFrame(out)


def safe_filename(s):
    value = re.sub(r'[^A-Za-z0-9_\-]+', '_', str(s)).strip('_')
    return (value or 'outcome')[:65]


def group_labeler(meta, var):
    lookup = (getattr(meta, 'variable_value_labels', {}) or {}).get(var, {}) or {}
    def fmt(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, str) and not v.strip():
            return np.nan
        try:
            label = lookup.get(v)
        except TypeError:
            label = None
        raw_string = str(v).strip()
        return f'{label} [{raw_string}]' if label is not None else raw_string
    return fmt


def prepare_dataset(raw, meta, group_var, pre_var, post_var, id_var=None):
    if not (group_var in raw and pre_var in raw and post_var in raw):
        raise ValueError('At least one selected variable is missing from the SAV file.')
    if pre_var == post_var:
        raise ValueError('Pre and post must be different variables.')
    if len({pre_var, post_var, group_var}) < 3:
        raise ValueError('Group, pre and post must be distinct variables.')
    if id_var is not None and id_var not in raw:
        raise ValueError('The selected participant ID variable was not found.')
    if id_var is not None and id_var in [group_var, pre_var, post_var]:
        raise ValueError('The ID variable must differ from the grouping and outcome columns.')
    if id_var is None:
        subjects = pd.Series([str(i) for i in range(1, len(raw) + 1)], index=raw.index)
    else:
        subjects = raw[id_var].astype('string').str.strip()
        subjects = subjects.replace('', pd.NA)
        if subjects.dropna().duplicated().any():
            ex = subjects[subjects.duplicated(keep=False)].dropna().iloc[0]
            raise ValueError(f'Duplicate participant ID found ({ex}). Wide-format data must have one row per participant.')
    before_pre = raw[pre_var]
    before_post = raw[post_var]
    pre = pd.to_numeric(before_pre, errors='coerce').replace([np.inf, -np.inf], np.nan)
    post = pd.to_numeric(before_post, errors='coerce').replace([np.inf, -np.inf], np.nan)
    groups = raw[group_var].map(group_labeler(meta, group_var))
    d = pd.DataFrame({'Subject': subjects, 'Group': groups, 'Pre': pre, 'Post': post}, index=raw.index)
    reasons = pd.DataFrame({
        'Missing ID': d['Subject'].isna(), 'Missing group': d['Group'].isna(),
        'Missing pre': d['Pre'].isna(), 'Missing post': d['Post'].isna()
    })
    complete = d.loc[~reasons.any(axis=1)].copy()
    if complete.empty:
        raise ValueError('No complete cases remain after excluding missing ID/group/pre/post.')
    if complete.Subject.duplicated().any():
        raise ValueError('Participant IDs are not unique in the final dataset.')
    levels = sorted(complete.Group.unique().tolist())
    if len(levels) < 2:
        raise ValueError('At least two between-subject groups with complete cases are required.')
    if len(levels) > 12:
        raise ValueError('More than 12 groups found. Verify the group variable is categorical, not a continuous score.')
    sizes = complete['Group'].value_counts()
    if sizes.min() < 3:
        raise ValueError('Every group requires at least three complete participants for this full diagnostic workflow. Current counts: ' + str(sizes.to_dict()))
    if complete[['Pre', 'Post']].to_numpy().astype(float).std() == 0:
        raise ValueError('All observed values are identical; ANOVA is undefined.')
    complete['MeanScore'] = (complete['Pre'] + complete['Post']) / 2
    complete['Change'] = complete['Post'] - complete['Pre']
    residual_change_ss = sum(float(((g.Change - g.Change.mean()) ** 2).sum())
                             for _, g in complete.groupby('Group'))
    residual_mean_ss = sum(float(((g.MeanScore - g.MeanScore.mean()) ** 2).sum())
                           for _, g in complete.groupby('Group'))
    if residual_change_ss == 0 or residual_mean_ss == 0:
        raise ValueError('One ANOVA error term has zero variance across participants within groups; the F tests cannot be estimated reliably.')
    sample = pd.DataFrame([
        {'Accounting item': 'Rows in SAV', 'Count': len(d)},
        {'Accounting item': 'Missing ID (may overlap other categories)', 'Count': int(reasons['Missing ID'].sum())},
        {'Accounting item': 'Missing group (may overlap other categories)', 'Count': int(reasons['Missing group'].sum())},
        {'Accounting item': 'Missing pre (may overlap other categories)', 'Count': int(reasons['Missing pre'].sum())},
        {'Accounting item': 'Missing post (may overlap other categories)', 'Count': int(reasons['Missing post'].sum())},
        {'Accounting item': 'Excluded: at least one selected field missing', 'Count': int(reasons.any(axis=1).sum())},
        {'Accounting item': 'Complete participants analysed', 'Count': len(complete)}
    ])
    return complete, sample


def mixed_anova_two_times(d):
    """Split-plot ANOVA for one group factor and exactly two repeated times.

    Between test: one-way ANOVA of subject means (factor of two re-scales SS).
    Within and interaction: Type III/sum-coded model of subject change scores
    (factor of one-half re-scales SS). These are the mixed-ANOVA tests with
    separate between-person and within-person error terms. The time main effect
    estimates equal-weighted group marginal means for unbalanced groups.
    """
    groups = d['Group'].nunique()
    n = len(d)
    if n <= groups:
        raise ValueError('Insufficient residual degrees of freedom.')
    between_fit = ols('MeanScore ~ C(Group, Sum)', data=d).fit()
    change_fit = ols('Change ~ C(Group, Sum)', data=d).fit()
    bt = sm.stats.anova_lm(between_fit, typ=3)
    wt = sm.stats.anova_lm(change_fit, typ=3)
    bterm = next(x for x in bt.index if x.startswith('C(Group,'))
    wterm = next(x for x in wt.index if x.startswith('C(Group,'))
    parts = [
        ('Group', bt.loc[bterm], bt.loc['Residual'], 2.),
        ('Time', wt.loc['Intercept'], wt.loc['Residual'], 0.5),
        ('Group × Time', wt.loc[wterm], wt.loc['Residual'], 0.5)
    ]
    rows = []
    for label, effect, error, factor in parts:
        ss = float(effect['sum_sq']) * factor
        errss = float(error['sum_sq']) * factor
        df1, df2 = float(effect['df']), float(error['df'])
        fvalue = float(effect['F'])
        pvalue = float(effect['PR(>F)'])
        eta = ss / (ss + errss) if ss + errss > 0 else np.nan
        rows.append({
            'Source': label, 'SS': ss, 'df1': df1, 'df2': df2,
            'MS': ss / df1, 'Error SS': errss, 'Error MS': errss / df2,
            'F': fvalue, 'p': pvalue, 'Partial eta squared': eta,
            'Significant (uncorrected)': None
        })
    result=pd.DataFrame(rows)
    check=stats.f_oneway(*[g['Change'].to_numpy() for _,g in d.groupby('Group',sort=True)])
    observed=float(result.loc[result.Source=='Group × Time','F'].iloc[0])
    if np.isfinite(check.statistic) and not np.isclose(observed,check.statistic,rtol=1e-7,atol=1e-7):
        raise RuntimeError('Internal numerical check failed for the group × time interaction.')
    return result


def stats_ci(arr, alpha=0.05):
    x = np.asarray(arr, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    mean = float(np.mean(x)) if n else np.nan
    sd = float(np.std(x, ddof=1)) if n > 1 else np.nan
    se = sd / math.sqrt(n) if n > 1 else np.nan
    crit = stats.t.ppf(1-alpha/2, n-1) if n > 1 else np.nan
    return dict(N=n, Mean=mean, SD=sd, SE=se,
                CI_Lower=mean-crit*se if n > 1 else np.nan,
                CI_Upper=mean+crit*se if n > 1 else np.nan)


def descriptives(d, alpha):
    desc, changes = [], []
    for label, sub in d.groupby('Group', sort=True):
        for time in ['Pre', 'Post']:
            desc.append({'Group': label, 'Time': time, **stats_ci(sub[time], alpha)})
        delta = stats_ci(sub['Change'], alpha)
        dz = delta['Mean']/delta['SD'] if delta['SD'] > 0 else np.nan
        changes.append({'Group': label, **delta, 'Cohen dz (paired)': dz})
    return pd.DataFrame(desc), pd.DataFrame(changes)


def assumption_tests(d):
    rows = []
    sizes = d['Group'].value_counts().to_dict()
    rows.append({'Diagnostic': 'Independence of participants', 'Group/time': 'All', 'Statistic': np.nan,
                 'p': np.nan, 'Interpretation': 'Cannot be tested statistically; establish from study design.'})
    rows.append({'Diagnostic': 'Sphericity', 'Group/time': 'All', 'Statistic': np.nan,
                 'p': np.nan, 'Interpretation': 'Automatically satisfied with exactly two time points.'})
    for label, sub in d.groupby('Group', sort=True):
        x = sub['Change'].to_numpy(dtype=float)
        if 3 <= len(x) <= 5000 and np.std(x, ddof=1) > 0:
            with np.errstate(all='ignore'):
                try:
                    s, p = stats.shapiro(x)
                    note = 'Shapiro–Wilk for within-person change; p < alpha suggests non-normality.'
                except Exception as e:
                    s, p, note = np.nan, np.nan, f'Not available: {type(e).__name__}'
        else:
            s, p = np.nan, np.nan
            note = 'Not computed: N > 5000, N < 3, or no within-group change-score variance.'
        rows.append({'Diagnostic': 'Shapiro–Wilk: change scores', 'Group/time': label,
                     'Statistic': s, 'p': p, 'Interpretation': note})
    for col, context in [('MeanScore','Subject mean (group-effect error term)'),
                         ('Change','Change score (time and group × time error term)')]:
        arrays = [sub[col].to_numpy(dtype=float) for _, sub in d.groupby('Group', sort=True)]
        if all(np.std(x, ddof=1) == 0 for x in arrays):
            st, p = np.nan, np.nan
            note = 'Not meaningful because all groups have zero within-group variance.'
        else:
            try:
                st, p = stats.levene(*arrays, center='median')
                note = 'Brown–Forsythe (median-centred Levene); small p suggests unequal group variances.'
            except Exception as e:
                st, p, note = np.nan, np.nan, f'Not available: {type(e).__name__}'
        rows.append({'Diagnostic': 'Brown–Forsythe variance homogeneity',
                     'Group/time': context, 'Statistic': st, 'p': p, 'Interpretation': note})
    # Tukey fences on within-group change scores: the same rule as the change boxplots.
    # (A |z| >= 3 rule cannot flag anything in groups of 10 or fewer: with the sample SD,
    # the largest possible |z| is (n - 1) / sqrt(n), e.g. 2.85 for n = 10.)
    flags = []
    for label, sub in d.groupby('Group', sort=True):
        q1, q3 = np.percentile(sub['Change'].to_numpy(dtype=float), [25, 75])
        lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        atypical = sub.loc[(sub['Change'] < lower) | (sub['Change'] > upper),
                           ['Subject','Group','Pre','Post','Change']].copy()
        atypical['Lower fence'] = lower
        atypical['Upper fence'] = upper
        if not atypical.empty:
            flags.append(atypical)
    outliers = pd.concat(flags, ignore_index=True) if flags else pd.DataFrame(
        columns=['Subject','Group','Pre','Post','Change','Lower fence','Upper fence'])
    rows.append({'Diagnostic': 'Flagged outside Tukey fences (1.5 × IQR) on change', 'Group/time': 'All',
                 'Statistic': len(outliers), 'p': np.nan,
                 'Interpretation': 'Potential outliers flagged only; NOT automatically excluded.'})
    return pd.DataFrame(rows), outliers


def welch_comparison(a, b, alpha):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    ma, mb = np.mean(a), np.mean(b)
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    v1, v2 = va/na, vb/nb
    se = math.sqrt(v1+v2)
    df_num = (v1+v2)**2
    df_den = v1**2/(na-1) + v2**2/(nb-1)
    dfree = df_num/df_den if df_den > 0 else np.nan
    estimate = ma-mb
    if se > 0 and np.isfinite(dfree):
        t = estimate/se
        p = float(2*stats.t.sf(abs(t), dfree))
        crit = stats.t.ppf(1-alpha/2, dfree)
        lo, hi = estimate-crit*se, estimate+crit*se
    else:
        t = p = lo = hi = np.nan
    pooled_var = ((na-1)*va+(nb-1)*vb)/(na+nb-2)
    pooled_sd = math.sqrt(pooled_var) if pooled_var > 0 else np.nan
    hedges_g = (estimate/pooled_sd)*(1-3/(4*(na+nb)-9)) if pooled_sd > 0 else np.nan
    return dict(Estimate=estimate, SE=se, df=dfree, t=t,
                p=p, CI_Lower=lo, CI_Upper=hi, Effect_size=hedges_g)


def morris_dppc2(group_a, group_b):
    """Morris (2008) d_ppc2: difference in mean change divided by the pooled pre-test SD,
    with small-sample bias correction. Unlike a d standardized by change-score SD, it is on
    the same scale as between-group effect sizes for the raw outcome."""
    na, nb = len(group_a), len(group_b)
    if na < 2 or nb < 2:
        return np.nan
    pooled_pre = math.sqrt(((na-1)*np.var(group_a['Pre'], ddof=1) +
                            (nb-1)*np.var(group_b['Pre'], ddof=1)) / (na+nb-2))
    if not pooled_pre > 0:
        return np.nan
    cp = 1 - 3/(4*(na+nb-2) - 1)
    return cp * (group_a['Change'].mean() - group_b['Change'].mean()) / pooled_pre


def pairwise_tests(d, alpha):
    rows = []
    groups = list(sorted(d.Group.unique()))
    for group in groups:
        sub = d.loc[d.Group == group]
        x = sub.Change.to_numpy(dtype=float)
        summary = stats_ci(x, alpha)
        sd, n = summary['SD'], summary['N']
        se = summary['SE']
        t = summary['Mean']/se if se > 0 else np.nan
        p = float(2*stats.t.sf(abs(t), n-1)) if np.isfinite(t) else np.nan
        effect = summary['Mean']/sd if sd > 0 else np.nan
        rows.append({'Family': 'Within-group paired pre/post', 'Comparison': f'{group}: Post − Pre',
                     'Test': 'Paired t', 'N1': n, 'N2': np.nan,
                     'Estimate': summary['Mean'], 'SE': se, 'df': n-1,
                     't': t, 'p': p, 'CI_Lower': summary['CI_Lower'],
                     'CI_Upper': summary['CI_Upper'], 'Effect_size': effect,
                     'Effect_size_type': 'Cohen dz'})
    for time in ['Pre','Post']:
        for a,b in combinations(groups,2):
            x = d.loc[d.Group == a, time]
            y = d.loc[d.Group == b, time]
            tst = welch_comparison(x,y,alpha)
            rows.append({'Family': 'Between-group at each time',
                         'Comparison': f'{time}: {a} − {b}', 'Test': 'Welch independent t',
                         'N1': len(x), 'N2': len(y), **tst,
                         'Effect_size_type': 'Hedges g (pooled SD)'})
    for a,b in combinations(groups,2):
        x = d.loc[d.Group == a,'Change']
        y = d.loc[d.Group == b,'Change']
        tst = welch_comparison(x,y,alpha)
        rows.append({'Family': 'Between-group change contrasts',
                     'Comparison': f'Change: {a} − {b}', 'Test': 'Welch independent t',
                     'N1': len(x), 'N2': len(y), **tst,
                     'Effect_size_type': 'Hedges g (pooled SD of change scores)',
                     'Effect_size_2': morris_dppc2(d.loc[d.Group == a], d.loc[d.Group == b]),
                     'Effect_size_2_type': 'Morris d_ppc2 (pooled pre-test SD)'})
    df = pd.DataFrame(rows)
    df['p_Holm'] = np.nan
    df['Significant after Holm'] = False
    for family, indices in df.groupby('Family').groups.items():
        valid = [i for i in indices if pd.notna(df.loc[i,'p'])]
        if valid:
            reject, corrected, _, _ = multipletests(df.loc[valid,'p'].to_numpy(),
                                                      alpha=alpha, method='holm')
            df.loc[valid,'p_Holm'] = corrected
            df.loc[valid,'Significant after Holm'] = reject
    return df


def make_plots(d, desc, changes, outpath, title, alpha):
    outpath = Path(outpath)
    outpath.mkdir(parents=True, exist_ok=True)
    paths=[]
    groups = sorted(d.Group.unique())
    fig, ax = plt.subplots(figsize=(9,6))
    for label in groups:
        sub = desc.loc[desc.Group == label].set_index('Time').reindex(['Pre','Post'])
        y = sub.Mean.to_numpy(dtype=float)
        lo = sub.CI_Lower.to_numpy(dtype=float)
        hi = sub.CI_Upper.to_numpy(dtype=float)
        ax.errorbar([0,1], y, yerr=[y-lo,hi-y], marker='o', capsize=5,
                    linewidth=2, label=f'{label} (n={int(sub.N.iloc[0])})')
    ax.set_xticks([0,1], ['Pre','Post'])
    ax.set_ylabel('Observed outcome score')
    ax.set_title(title+' — cell means with confidence intervals')
    ax.legend(loc='best', fontsize=9)
    ax.grid(axis='y', alpha=.25)
    fig.tight_layout()
    path=outpath/'01_Interaction_Means_CI.png';fig.savefig(path,dpi=180,bbox_inches='tight');plt.close(fig);paths.append(path)
    fig,ax=plt.subplots(figsize=(max(8,len(groups)*1.2),5.5))
    dataset=[d.loc[d.Group==g,'Change'].to_numpy(dtype=float) for g in groups]
    ax.boxplot(dataset, tick_labels=groups, showmeans=True)
    ax.axhline(0,color='gray',linewidth=1,linestyle='--')
    ax.tick_params(axis='x', rotation=20)
    ax.set_ylabel('Post − Pre')
    ax.set_title(title+' — individual change by group')
    fig.tight_layout()
    path=outpath/'02_Change_Boxplots.png';fig.savefig(path,dpi=180,bbox_inches='tight');plt.close(fig);paths.append(path)
    for k,group in enumerate(groups,1):
        fig,ax=plt.subplots(figsize=(6,5.5))
        stats.probplot(d.loc[d.Group==group,'Change'],dist='norm',plot=ax)
        ax.set_title(f'{title}: change-score Q–Q | {group}')
        fig.tight_layout()
        path=outpath/f'03_QQ_Group_{k:02d}.png';fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig);paths.append(path)
    return paths


def format_p(p):
    if not np.isfinite(p): return 'not estimable'
    return 'p < .001' if p < .001 else f'p = {p:.3f}'


def make_excel(path, tabs, alpha):
    with pd.ExcelWriter(path, engine='xlsxwriter',
                        datetime_format='yyyy-mm-dd hh:mm') as writer:
        book=writer.book
        headline=book.add_format({'bold':True,'font_color':'white', 'bg_color':'#243B53',
                                  'font_size':14,'valign':'vcenter'})
        column_fmt=book.add_format({'bold':True,'font_color':'white','bg_color':'#486581',
                                    'border':0,'text_wrap':True,'valign':'vcenter'})
        for sheet_name, frame in tabs.items():
            sheet_name=sheet_name[:31]
            if frame is None: continue
            copy=frame.copy()
            for col in copy.select_dtypes(include=['object','string']).columns:
                copy[col] = copy[col].map(lambda z: str(z)[:32000] if pd.notna(z) else '')
            if copy.shape[1] == 0:
                copy = pd.DataFrame({'Note':['No rows available']})
            copy.to_excel(writer,sheet_name=sheet_name,index=False,startrow=2)
            sh=writer.sheets[sheet_name]
            sh.set_row(0,29,headline)
            sh.write(0,0,sheet_name.replace('_',' '),headline)
            sh.set_row(2,29,column_fmt)
            for j, c in enumerate(copy.columns):
                sh.write(2,j,str(c),column_fmt)
                max_content = max([len(str(c))] + [len(str(v)) for v in copy[c].head(100).values])
                sh.set_column(j,j,min(48,max(12,max_content+2)),
                              book.add_format({'num_format':'0.0000'}) if pd.api.types.is_float_dtype(copy[c]) else None)
            sh.freeze_panes(3,0)
            if len(copy)>0:
                sh.autofilter(2,0,len(copy)+2,len(copy.columns)-1)
                for j,col in enumerate(copy.columns):
                    if col in ('p','p_Holm'):
                        sh.conditional_format(3,j,len(copy)+2,j,
                                              {'type':'cell','criteria':'<','value':alpha,
                                               'format':book.add_format({'bg_color':'#DFF4E5'})})


def add_word_table(doc, frame, columns, max_rows=60):
    selected = frame.loc[:, [x for x in columns if x in frame.columns]].head(max_rows)
    table=doc.add_table(rows=1,cols=len(selected.columns))
    table.style='Table Grid'
    for j,c in enumerate(selected.columns):
        cell=table.rows[0].cells[j]
        cell.text=c
        for p in cell.paragraphs:
            for run in p.runs: run.bold=True
    for _,row in selected.iterrows():
        cells=table.add_row().cells
        for j,value in enumerate(row):
            if pd.isna(value): text='—'
            elif isinstance(value,(float,np.floating)): text=f'{value:.3f}'
            else: text=str(value)
            cells[j].text=text
    return table


def make_word(path, title, filename, grp, pre, post, id_col, alpha, sample,
              desc, changes, anova, assumptions, pairs, outliers, figures,
              caveats):
    doc=Document()
    sect=doc.sections[0]
    sect.top_margin=Inches(.75);sect.bottom_margin=Inches(.75)
    styles=doc.styles
    styles['Normal'].font.name='Aptos'; styles['Normal'].font.size=Pt(9)
    doc.add_heading('Mixed-design ANOVA report',0)
    doc.add_paragraph(title,style='Subtitle')
    doc.add_paragraph('Generated '+datetime.now().strftime('%Y-%m-%d %H:%M')+
                      ' | Source: '+Path(filename).name)
    doc.add_heading('Design and analysis choices',1)
    doc.add_paragraph(f'Between-subject factor: {grp}. Within-subject factor: time (Pre, Post). '
                      f'SPSS outcomes: {pre} and {post}. Participant identifier: {id_col or "generated row ID"}. '
                      f'Alpha = {alpha:g}; confidence interval level = {100*(1-alpha):.1f}%.')
    doc.add_paragraph('Two-level split-plot mixed ANOVA. The between-subject group main effect is '
                      'tested using subject means; time and group × time are tested using subject '
                      'change scores with their own within-subject error term. Sum-to-zero contrasts '
                      'and Type III tests are used; the time main effect compares equal-weighted '
                      'group marginal changes where group sizes differ. Participants with missing '
                      'ID, group, pre or post are excluded listwise. No missing-data imputation.')
    doc.add_paragraph('Which question this answers: whether mean pre-to-post change differs between groups. '
                      'If groups were randomized, adjusting post-test scores for pre-test scores (ANCOVA) '
                      'estimates the same effect with more power. If groups differed before the study, '
                      'such as clinical groups, change-score analysis and ANCOVA answer different questions '
                      "and can disagree (Lord's paradox); choose between them on the research question, "
                      'before looking at results.')
    doc.add_heading('Participants and missing data',1)
    add_word_table(doc,sample,['Accounting item','Count'],25)
    doc.add_paragraph('The missing-field rows overlap: do not sum their counts to obtain excluded N.')
    doc.add_heading('Descriptive statistics',1)
    add_word_table(doc,desc,['Group','Time','N','Mean','SD','CI_Lower','CI_Upper'])
    doc.add_paragraph('Confidence intervals for cell means are across participants at each time; '
                      'they do not represent uncertainty in within-person change or in an interaction.')
    doc.add_heading('Mixed ANOVA',1)
    add_word_table(doc,anova,['Source','df1','df2','F','p','Partial eta squared'])
    for _,row in anova.iterrows():
        df1,df2=int(row['df1']),int(row['df2'])
        direction='statistically significant' if row.p < alpha else 'not statistically significant'
        doc.add_paragraph(f"{row.Source}: F({df1}, {df2}) = {row.F:.3f}, "
                          f"{format_p(row.p)}, partial η² = {row['Partial eta squared']:.3f}; "
                          f'{direction} at alpha {alpha:g}.')
    interaction=float(anova.loc[anova.Source=='Group × Time','p'].iloc[0])
    doc.add_paragraph('The group × time test is the principal omnibus test of whether '
                      'average pre-to-post change differs between groups. '+
                      ('It provides evidence of a difference in change.' if interaction<alpha else
                       'It did not provide sufficient evidence of a difference in change; this does not establish equivalence.'))
    doc.add_heading('Assumptions and checks',1)
    add_word_table(doc,assumptions,['Diagnostic','Group/time','Statistic','p'],65)
    doc.add_paragraph(f'Outlier flags based on Tukey fences (1.5 × IQR) on within-group change scores: {len(outliers)}; '
                      'none were automatically removed. Sphericity corrections are unnecessary '
                      'for exactly two repeated measurements. Shapiro–Wilk tests are descriptive '
                      'diagnostics, particularly sensitive with large samples; inspect Q–Q plots.')
    doc.add_heading('Follow-up comparisons',1)
    doc.add_paragraph('Exploratory paired within-group tests, Welch between-group tests at each '
                      'time, and Welch contrasts of changes are reported. Holm correction is '
                      'applied separately to each prespecified family of comparisons. Direction '
                      'is exactly as written in the Comparison column. These tests use different '
                      'variance assumptions from the pooled-error omnibus ANOVA.')
    add_word_table(doc,pairs,['Family','Comparison','Estimate','df','t','p','p_Holm','Effect_size','Effect_size_2'],max_rows=150)
    doc.add_paragraph('Effect sizes: Cohen dz for paired tests; Hedges g (pooled SD) for between-group '
                      'tests at each time. For change contrasts, Effect_size is Hedges g standardized by the '
                      'pooled SD of change scores, which exceeds a raw-score d whenever the pre-post '
                      'correlation is above .5; Effect_size_2 is Morris (2008) d_ppc2, standardized by the '
                      'pooled pre-test SD and comparable to between-group effect sizes on the outcome scale. '
                      'Changes use Post − Pre, so positive change indicates an increase.')
    doc.add_heading('Figures',1)
    for fig in figures[:2]:
        doc.add_paragraph(Path(fig).name)
        doc.add_picture(str(fig),width=Inches(6.05))
    doc.add_paragraph('Group-specific Q–Q figures are saved separately as PNG files.')
    doc.add_heading('Limitations and interpretation notes',1)
    for note in caveats: doc.add_paragraph(note,style='List Bullet')
    doc.add_paragraph('Documentation: https://www.statsmodels.org/stable/examples/notebooks/generated/interactions_anova.html; '
                      'https://github.com/Roche/pyreadstat')
    footer=sect.footer.paragraphs[0]
    footer.text='Generated reproducible analysis | Review against original study design before publication'
    doc.save(path)


def analyze_one(raw, meta, sav_name, group_var, pre_var, post_var, id_var, title,
                alpha, folder):
    d, sample = prepare_dataset(raw,meta,group_var,pre_var,post_var,id_var)
    aov = mixed_anova_two_times(d)
    aov['Significant (uncorrected)'] = aov['p'] < alpha
    descript, change = descriptives(d,alpha)
    assumptions, outliers = assumption_tests(d)
    pairs = pairwise_tests(d,alpha)
    folder=Path(folder)
    folder.mkdir(parents=True,exist_ok=True)
    pictures=make_plots(d,descript,change,folder,title,alpha)
    groups=d.Group.value_counts().sort_index()
    notes=[
       'Independence of participants and the appropriateness of the group variable cannot be tested from this dataset.',
       'If participants are clustered within schools, wards, sites, classrooms or other units, the analysis does not account for intracluster dependence; consider a multilevel model.',
       'Mixed ANOVA here uses complete cases and assumes common within-group error variances. With substantial or differential missingness, consider a longitudinal mixed-effects model and missing-data sensitivity analyses.',
       'This analysis does not include baseline covariates, pre-post measurement-error adjustments, or causal identification. An interaction alone is not sufficient evidence of a causal intervention effect.',
       'The time main effect uses Type III sum coding (equal-weighted group marginal changes when cell sizes differ). Interpret main effects cautiously if the interaction is present.',
       'Cell-mean error bars are confidence intervals for individual time-point means, not for within-participant differences.',
       'If testing several outcomes, Holm corrections are applied within each outcome only. No correction across outcomes is performed.',
       'Review SPSS missing-value definitions and all variable labels before analysis; SPSS-declared user-missing codes are treated as missing by default.'
    ]
    if groups.min()<10:
        notes.append('At least one group contains fewer than ten complete participants: normality checks and inference may be unstable.')
    if groups.nunique()>1:
        notes.append('Complete-case group sizes differ; Type III time contrasts use equally weighted group marginal means.')
    tabs={
       'Overview':pd.DataFrame({'Setting':['Outcome','Source SAV','Group variable','Pre column','Post column','ID column','Alpha','Confidence level','Complete N','Number of groups','Group sizes','Analysis','Multiplicity'],
                                'Value':[title,Path(sav_name).name,group_var,pre_var,post_var,id_var or 'generated row number',alpha,1-alpha,len(d),len(groups),str(groups.to_dict()),'2-time mixed ANOVA Type III / sum contrasts','Holm within each pairwise family only']}),
       'Sample_Accounting':sample,
       'Group_N':groups.rename_axis('Group').reset_index(name='N'),
       'Cell_Descriptives':descript,
       'Change_Descriptives':change,
       'ANOVA_Type_III':aov,
       'Assumption_Checks':assumptions,
       'Pairwise_Holm':pairs,
       'Outlier_Flags':outliers,
       'Selected_Variables':variable_dictionary(raw,meta).query('Variable in @vars',local_dict={'vars':[group_var,pre_var,post_var]+([id_var] if id_var else [])}),
       'Interpretation_Cautions':pd.DataFrame({'Note':notes}),
       'Software_Versions':pd.DataFrame({'Package':['Python','pandas','numpy','scipy','statsmodels','matplotlib'],
                                        'Version':[sys.version.split()[0],pd.__version__,np.__version__,scipy.__version__,statsmodels.__version__,matplotlib.__version__]})
    }
    xlsx=folder/'Statistical_Tables.xlsx'
    docx=folder/'Mixed_ANOVA_Report.docx'
    make_excel(xlsx,tabs,alpha)
    make_word(docx,title,sav_name,group_var,pre_var,post_var,id_var,alpha,sample,descript,change,aov,assumptions,pairs,outliers,pictures,notes)
    (folder/'Analysis_Notes.txt').write_text('\n'.join([
       'Two-time (pre/post) mixed ANOVA; one between-subject factor.',
       'Design: wide .sav; listwise deletion; no imputation.',
       'Primary test: Group × Time (equal mean change null).',
       'Type III sum coding. Time main = equal-weighted group mean change.',
       'Time/main and group×time error = variance of within-subject changes;',
       'between main error = variance of within-subject means.',
       'Change contrast followups Welch + Holm (separate family).',
       'Change-contrast effect sizes: Hedges g on change-score SD, and Morris (2008) d_ppc2 on pre-test SD.',
       'Outlier flags: Tukey fences (1.5 x IQR) on within-group change; flags are not exclusions.',
       'Randomized groups: ANCOVA has more power. Pre-existing groups: change and ANCOVA answer different questions.',
       'No cross-outcome p-value correction.',
       *notes
    ]), encoding='utf-8')
    return {'Outcome':title,'N':len(d),'Groups':len(groups),
            'Interaction_F':float(aov.loc[aov.Source=='Group × Time','F'].iloc[0]),
            'Interaction_p':float(aov.loc[aov.Source=='Group × Time','p'].iloc[0]),
            'Folder':str(folder)}

## 5. Choose variables and run your analysis

In [ ]:
import ipywidgets as widgets
import uuid
from IPython.display import display, clear_output, HTML

variables = variable_dictionary(raw_data, spss_meta)
print('Check the SPSS metadata below, then make your selections.')
display(variables[['Variable', 'SPSS label','Type','Measure','Missing (loaded)','Distinct nonmissing']].head(60))
if len(variables) > 60:
    print(f'Only the first 60 of {len(variables)} variables are previewed; all remain selectable.')

label_dict = dict(zip(variables['Variable'], variables['SPSS label']))

def label(var):
    text = str(label_dict.get(var, '') or '')
    return f'{var} — {text[:55]}' if text else var

all_options = [(label(c), c) for c in raw_data.columns]
numeric_columns = [c for c in raw_data.columns if pd.api.types.is_numeric_dtype(raw_data[c])]
if len(numeric_columns) < 2:
    raise ValueError('Fewer than two numeric columns found; this two-time-point ANOVA requires numeric outcomes.')
number_options = [('— select a numeric variable —', None)] + [(label(c),c) for c in numeric_columns]
likely_groups = [c for c in raw_data.columns if 2 <= raw_data[c].nunique(dropna=True) <= 12]
if likely_groups:
    suggested_group=likely_groups[0]
else:
    suggested_group=raw_data.columns[0]

wide = widgets.Layout(width='440px')
short = widgets.Layout(width='280px')
group_ui = widgets.Dropdown(options=all_options, value=suggested_group,
                            description='Group:', style={'description_width':'120px'},layout=wide)
id_ui = widgets.Dropdown(options=[('(Generate row IDs — one row per participant)',None)]+all_options,
                         description='Participant ID:', value=None,
                         style={'description_width':'120px'}, layout=wide)
alpha_ui = widgets.BoundedFloatText(value=0.05,min=0.001,max=0.20,step=0.01,
                                    description='Alpha:',style={'description_width':'120px'},
                                    layout=widgets.Layout(width='260px'))
rows=[]
pair_container=widgets.VBox([])

def refresh_pairs():
    pair_container.children=tuple(r['box'] for r in rows)

def add_pair(_=None):
    number=len(rows)+1
    outcome_title=widgets.Text(value=f'Outcome {number}',description='Name:',
                               style={'description_width':'55px'},layout=widgets.Layout(width='240px'))
    pre=widgets.Dropdown(options=number_options,value=None,description='Pre:',
                         style={'description_width':'45px'},layout=short)
    post=widgets.Dropdown(options=number_options,value=None,description='Post:',
                          style={'description_width':'45px'},layout=short)
    rm=widgets.Button(description='Remove',icon='trash',button_style='',
                      layout=widgets.Layout(width='110px'))
    row={'title':outcome_title,'pre':pre,'post':post}
    row['box']=widgets.HBox([outcome_title,pre,post,rm],
                            layout=widgets.Layout(flex_flow='row wrap'))
    def remove(_):
        if len(rows)<=1:
            print('At least one outcome pair must remain.')
            return
        rows.remove(row)
        refresh_pairs()
    rm.on_click(remove)
    rows.append(row)
    refresh_pairs()

add_pair_button=widgets.Button(description='Add another outcome pair',icon='plus')
add_pair_button.on_click(add_pair)
add_pair()
run_button=widgets.Button(description='Run analysis & save to Google Drive',
                          button_style='success',icon='play',
                          layout=widgets.Layout(width='360px',height='42px'))
result_output=widgets.Output(layout=widgets.Layout(border='1px solid #d6dde4',
                                                   padding='10px',width='100%'))

def on_run(_):
    with result_output:
        clear_output(wait=True)
        run_button.disabled=True
        try:
            if not REPORT_ROOT.exists():
                raise RuntimeError('Google Drive is not mounted. Run Section 3 before proceeding.')
            selected=[]
            for idx,r in enumerate(rows,1):
                before, after=r['pre'].value,r['post'].value
                if before is None or after is None:
                    raise ValueError(f'Outcome pair {idx}: please select both pre and post columns.')
                if before==after:
                    raise ValueError(f'Outcome pair {idx}: pre and post must be different columns.')
                name=r['title'].value.strip() or f'Outcome_{idx}'
                selected.append((name,before,after))
            pairs=[(b,a) for _,b,a in selected]
            if len(set(pairs))!=len(pairs):
                raise ValueError('A pre/post pair has been selected more than once. Remove the duplicate.')
            group=group_ui.value
            person=id_ui.value
            if group is None:
                raise ValueError('Select one between-subject grouping variable.')
            stamp=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:5]
            session=REPORT_ROOT/(safe_filename(Path(sav_filename).stem)+'_'+stamp)
            session.mkdir(parents=True,exist_ok=True)
            completed=[]
            failures=[]
            print(f'Output folder: {session}')
            for k,(name,before,after) in enumerate(selected,1):
                destination=session/(f'{k:02d}_'+safe_filename(name))
                print(f'\nAnalysing {k}/{len(selected)}: {name} [{before} → {after}]')
                try:
                    row=analyze_one(raw_data,spss_meta,sav_filename,group,
                                    before,after,person,name,alpha_ui.value,destination)
                    completed.append(row)
                    print(f"  Saved Excel, Word and figures. Complete N={row['N']}, "
                          f"interaction p={row['Interaction_p']:.4g}")
                except Exception as error:
                    failures.append({'Outcome':name,'Error':f'{type(error).__name__}: {error}'})
                    print('  FAILED:',str(error))
            if completed:
                index=pd.DataFrame(completed)
                make_excel(session/'00_Analysis_Index.xlsx',{'Analysis_Index':index},alpha_ui.value)
                display(index)
            if failures:
                (session/'Errors.txt').write_text('\n'.join(f"{x['Outcome']}: {x['Error']}" for x in failures),encoding='utf-8')
                print('\nSome analyses could not run; see Errors.txt.')
                display(pd.DataFrame(failures))
            if completed:
                print(f'\nDONE. Open Google Drive → My Drive → Mixed_ANOVA_Reports → {session.name}')
                print('Each outcome folder contains .xlsx, .docx, figures, and analysis notes.')
            else:
                print('No analysis succeeded. Correct the selections, then click Run again.')
        except Exception as error:
            print(f'Validation error: {error}')
        finally:
            run_button.disabled=False

run_button.on_click(on_run)

display(widgets.HTML('<h3>Analysis selection</h3><p>Each row is one dependent variable measured at pre and post. The grouping variable is shared across all selected outcomes.</p>'))
display(widgets.VBox([group_ui,id_ui,alpha_ui,
                      widgets.HTML('<b>Dependent variable pairs</b>'),pair_container,
                      add_pair_button,run_button,result_output]))

### Interpreting the saved reports

- **Primary inferential test:** `Group × Time`, which assesses whether mean change differs between groups. A significant within-group pre/post comparison **does not** itself establish that groups differ in response.
- **Effect sizes:** partial eta squared for each ANOVA effect; Cohen dz for paired follow-ups; Hedges g for between-group comparisons at each time. Change contrasts report both Hedges g on the change-score SD and Morris (2008) d_ppc2 on the pre-test SD; only the latter is comparable to ordinary between-group effect sizes.
- **Assumptions:** Shapiro–Wilk tests use within-subject change scores, and Brown–Forsythe tests assess equality of variance for the two error terms (subject means and change scores). The Q–Q and change-score plots require substantive inspection. Sphericity is automatically satisfied for two time points.
- **Output:** results are organized by uploaded file and run, then by selected outcome. Analysis logs identify complete-case N, software versions, and limitations. Outliers are flagged with Tukey fences (1.5 × IQR) on within-group change scores, the same rule as the boxplots; flags are **not** automatic exclusions.
- **Change scores or ANCOVA:** with randomized groups, ANCOVA (post-test adjusted for pre-test) has more power for the same question. With pre-existing groups, the two analyses answer different questions and can disagree (Lord's paradox).
- **Important limits:** this notebook does not fit ANCOVA, linear mixed-effects models, clustered/site-adjusted models, or models with three or more repeated measures. For unequal group sizes, the Type III time effect is an equal-weighted comparison of group marginal changes. Familywise adjustment is **not** applied across different selected outcomes.